In [7]:
from selenium import webdriver
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service 
import pandas as pd
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import os

s = Service("/mnt/0498342198341420/Chrome_Driver_Ubuntu/chromedriver")
driver = webdriver.Chrome(service = s)



In [8]:
#https://www.transfermarkt.com/manchester-city/startseite/verein/281/saison_id/2024
# Cài đặt Selenium
wait = WebDriverWait(driver, 15)
options = webdriver.ChromeOptions()
#options.add_argument('--headless')  # Bỏ nếu muốn thấy trình duyệt
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# Bước 1: Lấy link từng CLB trong EPL
link_club = []
driver.get('https://www.transfermarkt.com/bundesliga/startseite/wettbewerb/L1')
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.items")))

# Lấy tất cả các câu lạc bộ trong Premier League
tbody = driver.find_element(By.XPATH, '/html/body/div[1]/main/div[1]/div[1]/div[2]/div[2]/div/table/tbody')
rows = tbody.find_elements(By.TAG_NAME, 'tr')

for row in rows:
    try:
        
        link = row.find_element(By.CSS_SELECTOR, 'td.hauptlink.no-border-links a')
        #link = WebDriverWait(r, 10).until(
        #    EC.presence_of_element_located((By.CSS_SELECTOR, 'td.hauptlink.no-border-links a'))
        #)
        # Lấy tên CLB
        time.sleep(0.5)
        club_name = link.text.strip()
        print(club_name)
        link_href = link.get_attribute("href")
        link_club.append((link_href, club_name))
    except:
        continue




Bayern Munich
Bayer 04 Leverkusen
RB Leipzig
Borussia Dortmund
VfB Stuttgart
Eintracht Frankfurt
VfL Wolfsburg
TSG 1899 Hoffenheim
SC Freiburg
1.FSV Mainz 05
Borussia Mönchengladbach
SV Werder Bremen
FC Augsburg
1.FC Union Berlin
1.FC Heidenheim 1846
FC St. Pauli
VfL Bochum
Holstein Kiel


In [11]:
# Bước 2: Truy cập từng link của câu lạc bộ và lấy danh sách cầu thủ
for link_url, club in link_club:
    data = []
    if os.path.exists(club + '.csv'):
        print(f"Đã có file {club}.csv -> bỏ qua.")
        continue
    driver.get(link_url)
    #click = driver.find_element(By.XPATH, '/html/body/div/main/div[1]/div[1]/div/div[2]/a[2]/div/span').click()
    detailed_view_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, '/html/body/div/main/div[1]/div[1]/div/div[2]/a[2]/div/span'))
    )
    detailed_view_btn.click()
    wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    time.sleep(1)
    
    club_name = club
    try:
        table = wait.until(EC.presence_of_element_located((By.XPATH, '//table[contains(@class,"items")]/tbody')))
        rows = table.find_elements(By.XPATH, './/tr[contains(@class,"odd") or contains(@class,"even")]')
    except:
        print("⚠️ Không tìm thấy bảng cầu thủ.")
        continue

    print(f"🔢 Số cầu thủ: {len(rows)}")

    for r in rows:
        # td = r.find_elements(By.TAG_NAME, 'td')
        td = WebDriverWait(r, 10).until(
            EC.presence_of_all_elements_located((By.TAG_NAME, 'td'))
        )
        try:
            # Name, Height, Contract, Market Value, main_position
            # Name
            link_temp = WebDriverWait(r, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'td.hauptlink a'))
            )
            name = link_temp.text.strip()
            
            # Market Value
            price = WebDriverWait(r, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'td.rechts.hauptlink a'))
            )
            price = price.text.strip()

            # Main_position 
            main_position = td[4].text.strip()
            # Height 
            height = td[7].text.strip()
            # Contract 
            contract = td[11].text.strip()
            # Foot 
            foot = td[8].text.strip()
            data.append({
                'CLB': club_name,
                'Name': name,
                'Main_position' : main_position,
                'Height' : height,
                'Foot' : foot,
                'Contract' : contract, 
                'Market_Value' : price
            })
            
            print(name, price, main_position, height, contract, foot)
        except:
            print(" → [Không tìm thấy cầu thủ]")
            continue
    df = pd.DataFrame(data)
    df.to_csv(club_name + '.csv')



Đã có file Bayern Munich.csv -> bỏ qua.
Đã có file Bayer 04 Leverkusen.csv -> bỏ qua.
Đã có file RB Leipzig.csv -> bỏ qua.
Đã có file Borussia Dortmund.csv -> bỏ qua.
Đã có file VfB Stuttgart.csv -> bỏ qua.
Đã có file Eintracht Frankfurt.csv -> bỏ qua.
Đã có file VfL Wolfsburg.csv -> bỏ qua.
Đã có file TSG 1899 Hoffenheim.csv -> bỏ qua.
Đã có file SC Freiburg.csv -> bỏ qua.
Đã có file 1.FSV Mainz 05.csv -> bỏ qua.
Đã có file Borussia Mönchengladbach.csv -> bỏ qua.
Đã có file SV Werder Bremen.csv -> bỏ qua.
Đã có file FC Augsburg.csv -> bỏ qua.
Đã có file 1.FC Union Berlin.csv -> bỏ qua.
Đã có file 1.FC Heidenheim 1846.csv -> bỏ qua.
Đã có file FC St. Pauli.csv -> bỏ qua.
Đã có file VfL Bochum.csv -> bỏ qua.
Đã có file Holstein Kiel.csv -> bỏ qua.
